In [1]:
import numpy as np
import pandas as pd

In [2]:
df_temperature = pd.read_csv("Source_Data/temperaturdaten.csv", encoding="utf-8", sep=";")
df_fitness =  pd.read_csv("Dataframes/all_rides_in_one.csv", encoding="utf-8")

Der DataFrame df_temperature enthält Messdaten der Station München-Stadt für den Zeitraum vom 10. Februar 2025 bis zum 13. August 2025.

Jeder Datenpunkt umfasst die innerhalb der letzten zehn Minuten gemessene Höchst- und Tiefsttemperatur. Die Messungen wurden in einer Höhe von 2 Metern sowie 5 Zentimetern über dem Boden durchgeführt.

Für die weitere Analyse wird ausschließlich die Lufttemperatur in 2 Metern Höhe verwendet, da diese die Temperatur in meiner Wohnung vermutlich stärker beeinflusst. Die entsprechenden Werte sind in den Spalten TX_10 (Höchsttemperatur) und TN_10 (Tiefsttemperatur) enthalten.

In [3]:
df_temperature["TM_10"] = (df_temperature["TX_10"] + df_temperature["TN_10"])/2
print(df_temperature)

       STATIONS_ID    MESS_DATUM    QN  TX_10  TX5_10  TN_10  TN5_10  eor  \
0             3379  202502100000     3   -0.3    -999   -0.4    -1.5  eor   
1             3379  202502100010     3   -0.4    -999   -0.5    -2.0  eor   
2             3379  202502100020     3   -0.4    -999   -0.6    -2.1  eor   
3             3379  202502100030     3   -0.6    -999   -0.7    -2.3  eor   
4             3379  202502100040     3   -0.5    -999   -0.6    -2.0  eor   
...            ...           ...   ...    ...     ...    ...     ...  ...   
79195         3379  202608132310     2   18.0    -999   17.9    16.1  eor   
79196         3379  202608132320     2   17.9    -999   17.8    16.1  eor   
79197         3379  202608132330     2   17.9    -999   17.7    16.2  eor   
79198         3379  202608132340     2   17.7    -999   17.6    16.0  eor   
79199         3379  202608132350     2   17.6    -999   17.4    15.9  eor   

       TM_10  
0      -0.35  
1      -0.45  
2      -0.50  
3      -0.65  


In [4]:
df_temperature.drop(columns=['TX5_10', 'TN5_10', 'eor'], inplace=True)
print(df_temperature)

       STATIONS_ID    MESS_DATUM    QN  TX_10  TN_10  TM_10
0             3379  202502100000     3   -0.3   -0.4  -0.35
1             3379  202502100010     3   -0.4   -0.5  -0.45
2             3379  202502100020     3   -0.4   -0.6  -0.50
3             3379  202502100030     3   -0.6   -0.7  -0.65
4             3379  202502100040     3   -0.5   -0.6  -0.55
...            ...           ...   ...    ...    ...    ...
79195         3379  202608132310     2   18.0   17.9  17.95
79196         3379  202608132320     2   17.9   17.8  17.85
79197         3379  202608132330     2   17.9   17.7  17.80
79198         3379  202608132340     2   17.7   17.6  17.65
79199         3379  202608132350     2   17.6   17.4  17.50

[79200 rows x 6 columns]


Die Fitnessdateien enthalten für jede Sekunde eines Workouts einen Datenpunkt. Dabei werden zahlreiche Messwerte in unterschiedlichen Spalten gespeichert. Einige dieser Spalten werden von Zwift jedoch nicht befüllt und enthalten daher keine relevanten Daten.

Im Folgenden wird zunächst ein kurzer Überblick über die vorhandenen Spalten gegeben.


In [6]:
print(df_fitness.describe())

       accumulated_power  activity_type       altitude  ball_speed  \
count                0.0            0.0  480169.000000         0.0   
mean                 NaN            NaN      93.575878         NaN   
std                  NaN            NaN     174.887729         NaN   
min                  NaN            NaN      -6.400000         NaN   
25%                  NaN            NaN      13.000000         NaN   
50%                  NaN            NaN      31.000000         NaN   
75%                  NaN            NaN      86.600000         NaN   
max                  NaN            NaN    1543.000000         NaN   

             cadence  cadence256  calories  combined_pedal_smoothness  \
count  480169.000000         0.0       0.0                        0.0   
mean       78.008318         NaN       NaN                        NaN   
std        20.136798         NaN       NaN                        NaN   
min         0.000000         NaN       NaN                        NaN   
25% 

In [7]:
print(df_fitness.count() == 0)

source_file                         False
accumulated_power                    True
activity_type                        True
altitude                            False
ball_speed                           True
cadence                             False
cadence256                           True
calories                             True
combined_pedal_smoothness            True
compressed_accumulated_power         True
compressed_speed_distance            True
core_data_quality                    True
core_reserved                        True
cycle_length                         True
cycles                               True
device_index                         True
distance                             True
enhanced_altitude                   False
enhanced_speed                      False
fractional_cadence                   True
gps_accuracy                         True
grade                                True
heart_rate                          False
heat_strain_index                 

In [ ]:
df_fitness.dropna(axis=1, how='all', inplace = True)
print(df_fitness.head())

In [ ]:
print(df_fitness[df_fitness['altitude'] != df_fitness['enhanced_altitude']])
print(df_fitness['speed_1s'].unique())

In [ ]:
df_fitness.drop(columns = ['enhanced_altitude', 'speed_1s'], inplace = True)
print(df_fitness.head())

In [ ]:
df_fitness.info()

In [ ]:
df_temperature.info()

In [ ]:
df_temperature.columns

In [ ]:
df_temperature.columns = df_temperature.columns.str.strip()
df_temperature.columns

In [ ]:
# 1. Datumsformate in echte datetime-Objekte umwandeln
df_temperature['MESS_DATUM'] = pd.to_datetime(df_temperature['MESS_DATUM'], format='%Y%m%d%H%M')
df_fitness['timestamp'] = pd.to_datetime(df_fitness['timestamp'], format='%Y-%m-%d %H:%M:%S')

# 2. Beide DataFrames nach der Zeit sortieren (Voraussetzung für merge_asof)
df_temperature = df_temperature.sort_values('MESS_DATUM')
df_fitness = df_fitness.sort_values('timestamp')

# 3. merge_asof: findet für jeden fitness-Zeitpunkt den passenden
#    MESS_DATUM-Wert, der <= timestamp ist (backward), 
#    aber maximal 10 Minuten zurückliegt (tolerance)
df_merged = pd.merge_asof(
    df_fitness,
    df_temperature[['MESS_DATUM', 'QN', 'TM_10']],
    left_on='timestamp',
    right_on='MESS_DATUM',
    direction='forward',
    tolerance=pd.Timedelta('10min')
)

In [ ]:
df_merged.sample(10)

In [ ]:
df_merged[df_merged.timestamp == '2026-01-10 12:40:00']

In [ ]:
df_merged.describe()

In [ ]:
df_merged[df_merged.TM_10 == -999]

In [ ]:
df_merged["heart_rate"] = df_merged["heart_rate"].replace(0, np.nan)
df_merged["TM_10"] = df_merged["TM_10"].replace(-999, np.nan)
df_merged["target_power"] = df_merged["target_power"].replace(0, np.nan)

In [ ]:
df_merged.describe()

In [ ]:
df_merged[df_merged.power >= 700]